# EXA-STAR: pre-train the seed architecture (Kaggle T4)

The evolutionary search stalled at ~MSE 0.99 because it started from a shallow (~1-layer),
untrained seed: a warm-started new layer can't out-train the shallow incumbent within one genome's
tiny budget, so evolution can't grow into depth. The fix is to start the search from a **capable,
pre-trained base**.

This notebook builds the deeper seed (a BrainLM-shaped `encoder_depth`+`decoder_depth` self-attention
stack, still ~100x smaller than BrainLM) and **fully trains it once** on real fMRI -- warmup+cosine
LR, periodic validation, early stopping, best-validation checkpoint -- saving the trained seed to
`PRETRAINED_SEED_PATH`. Then run `exa_vit_mae_evolved_kaggle.ipynb`, which loads this seed and
evolves architectural refinements from a model that already reconstructs.

> **Resumable:** if `PRETRAINED_SEED_PATH` already exists, this notebook *continues* training it
> (each session re-warms and keeps improving the saved best), so a Kaggle timeout mid-pre-train
> never loses progress. Watch the printed `val R2` climb; stop when it plateaus.
>
> **Reuse the SAME split + stats the evolution run will use** -- point `SPLIT_PATH`/`STATS_PATH` at
> the exact files (kept on Kaggle Persistence), so the seed is trained on the same data split the
> search and final evaluation use.

## 1. Configuration

In [ ]:
import os

# --- repo + data locations (EDIT; must match your evolution notebook) ---
REPO_PATH = "/kaggle/working/exa-star"
REPO_URL = "https://github.com/axj2613/exa-ae.git"     # your fork; add a token if private
REPO_BRANCH = "autoencoder-aryan"
HCP_ROOT = "/kaggle/working/hcp_complete"
ATLAS_COORDS = os.path.join(REPO_PATH, "datasets/hcp/atlases/A424_Coordinates.dat")

WORKDIR = "/kaggle/working"
SPLIT_PATH = os.path.join(WORKDIR, "subject_split.json")
STATS_PATH = os.path.join(WORKDIR, "norm_stats.npz")
LENGTH_INDEX_PATH = os.path.join(WORKDIR, "length_index.json")
PRETRAINED_SEED_PATH = os.path.join(WORKDIR, "pretrained_seed.pkl")   # <- the evolution notebook loads this

# --- model / seed architecture (MUST match the evolution notebook's config) ---
# WINDOW_LENGTH == TIME_PATCH_SIZE => a SINGLE temporal patch per parcel (424 tokens). The HCP
# signal decorrelates within ~5-10 TRs, so multiple 20-TR patches are temporally uncorrelated and
# only DILUTE the spatial (functional-connectivity) signal -- the real, learnable structure here.
# With one patch the task is "reconstruct masked parcels from visible ones," which trains cleanly
# (measured R^2 climbing toward the ~0.22 linear ceiling) instead of collapsing to the mean.
WINDOW_LENGTH = 20
TIME_PATCH_SIZE = 20
MASK_RATIO = 0.5             # BrainLM's ratio; more visible parcels => stronger reconstruction signal
SPLIT_RATIOS = (0.7, 0.1, 0.2)
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
DROPOUT = 0.1
SEED_ENCODER_DEPTH = 3        # encoder self-attention layers (last is the depth-0.5 bottleneck)
SEED_DECODER_DEPTH = 2        # decoder self-attention layers -- 3+2 ~= 1M params, ~100x < BrainLM

# --- pre-training schedule (the long one-time base-model training run) ---
PRETRAIN_STEPS = 15000        # raise if val R2 is still climbing at the end
PRETRAIN_BATCH = 64          # only 424 tokens now (single patch), so a large batch fits easily on a T4
PRETRAIN_LR = 1e-3
PRETRAIN_MIN_LR = 1e-5
PRETRAIN_WARMUP = 500
PRETRAIN_VAL_EVERY = 500
PRETRAIN_VAL_BATCHES = 32
PRETRAIN_PATIENCE = 12        # stop after this many val checks without improvement
TEST_BATCHES = 64            # batches for the final held-out test R^2

## 2. Fetch the repo + imports

In [ ]:
import subprocess
import sys

if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "origin", REPO_BRANCH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import pickle
from types import SimpleNamespace
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

from time_series.hcp_window_dataset import HCPWindowDataset
from genomes.vision_transformer_block_genome import VisionTransformerBlockGenome
from weight_generators.lamarckian_block_weight_generator import LamarckianBlockWeightGenerator
from evaluation_scripts.train_final_model import full_train, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 3. Dataset (same split + normalization as the search)

In [ ]:
dataset = HCPWindowDataset(
    root_dir=HCP_ROOT, atlas_coordinates_filename=ATLAS_COORDS,
    window_length=WINDOW_LENGTH, split_ratios=SPLIT_RATIOS,
    split_path=SPLIT_PATH, stats_path=STATS_PATH, length_index_path=LENGTH_INDEX_PATH,
)
print("parcels:", dataset.num_parcels, "| split sizes:", {k: len(v) for k, v in dataset.splits.items()})

## 4. Build (or resume) the deeper seed

In [ ]:
# resume-friendly: continue improving a previously-saved partial seed if present, else build fresh
if os.path.exists(PRETRAINED_SEED_PATH):
    with open(PRETRAINED_SEED_PATH, "rb") as seed_file:
        seed = pickle.load(seed_file)
    assert seed.window_length == WINDOW_LENGTH, "pretrained seed window_length != WINDOW_LENGTH"
    print(f"continuing pre-training from existing {PRETRAINED_SEED_PATH}")
else:
    seed = VisionTransformerBlockGenome(
        generation_number=0, num_parcels=dataset.num_parcels, window_length=WINDOW_LENGTH,
        parcel_coordinates=dataset.parcel_coordinates, d_model=D_MODEL, num_heads=NUM_HEADS,
        d_ff=D_FF, dropout=DROPOUT, time_patch_size=TIME_PATCH_SIZE, mask_ratio=MASK_RATIO,
        encoder_depth=SEED_ENCODER_DEPTH, decoder_depth=SEED_DECODER_DEPTH,
        weight_generator=LamarckianBlockWeightGenerator(),
    )
    print("built a fresh deeper seed")

# mark node/edge reachability (node.active) before any forward pass. Recent code does this in the
# genome constructor, but call it explicitly so this notebook works even against an older module
# version still cached in the kernel; it is idempotent.
seed.calculate_reachability()

seed.to(device)
report = seed.parameter_report()
print(f"seed size: {report['total_active_parameters']:,} params "
      f"(vs BrainLM's 111M) | layers by region: {report['node_type_counts_by_region']}")

## 5. Pre-train the seed

Long, dedicated training with warmup+cosine LR, periodic validation, early stopping, and
best-validation checkpointing to `PRETRAINED_SEED_PATH`. Watch `val R2` climb off 0 -- that is the
model learning to reconstruct. If it is still climbing at the last step, raise `PRETRAIN_STEPS` (or
just re-run this notebook, which continues from the saved seed).

In [ ]:
args = SimpleNamespace(
    output=PRETRAINED_SEED_PATH, lr=PRETRAIN_LR, min_lr=PRETRAIN_MIN_LR, warmup_steps=PRETRAIN_WARMUP,
    total_steps=PRETRAIN_STEPS, batch_size=PRETRAIN_BATCH, val_every=PRETRAIN_VAL_EVERY,
    val_batches=PRETRAIN_VAL_BATCHES, patience=PRETRAIN_PATIENCE,
)
full_train(seed, dataset, device, args)   # saves the best-validation seed to PRETRAINED_SEED_PATH

## 6. Held-out TEST R^2 of the pre-trained seed

In [ ]:
with open(PRETRAINED_SEED_PATH, "rb") as seed_file:
    best = pickle.load(seed_file)
best.to(device)
test_mse, test_r2 = evaluate(best, dataset, device, "test", PRETRAIN_BATCH, TEST_BATCHES, device.type == "cuda")
print(f"pre-trained seed held-out TEST reconstruction R^2 = {test_r2:.4f}  (MSE {test_mse:.5f})")
print("BrainLM reference: ~0.28 (HCP). The SEARCH then evolves refinements from this base.")

from IPython.display import FileLink, display
display(FileLink(os.path.relpath(PRETRAINED_SEED_PATH, WORKDIR)))

## Next step

Run `exa_vit_mae_evolved_kaggle.ipynb` with a matching `PRETRAINED_SEED_PATH`. On a fresh start it
loads this trained seed as the population's seed genome, so every evolved genome inherits the
working reconstruction model (Lamarckian) and the search optimizes architecture from a real signal
instead of stalling at the baseline. Keep `pretrained_seed.pkl`, `subject_split.json`, and
`norm_stats.npz` on Kaggle Persistence so the seed, split, and normalization stay consistent.